# Starter

In [ ]:
%cd /content/drive/MyDrive/GenAI/Unstructured and Multimodal Data/Multimodal with Gemini

/content/drive/MyDrive/GenAI/Unstructured and Multimodal Data/Multimodal with Gemini


In [ ]:
!pip install -q google-genai chromadb pymupdf Pillow requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:
import os
import pathlib
import mimetypes
import requests
import fitz  # PyMuPDF
import chromadb

from google import genai
from google.genai import types
from google.colab import userdata
from chromadb.config import Settings
from openai import OpenAI

In [ ]:
# -----------------------------
# API keys
# -----------------------------
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY2")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY3")

client = genai.Client(api_key=GEMINI_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# -----------------------------
# Models
# -----------------------------
EMBEDDING_MODEL = "gemini-embedding-2"
GEMINI_TEXT_MODEL = "gemini-3.5-flash-lite"
OPENAI_MODEL = "gpt-5.6-luna"

# 768 is smaller and faster than the default 3072 dimensions.
EMBEDDING_DIM = 768

# -----------------------------
# ChromaDB
# -----------------------------
COLLECTION_NAME = "multimodal_rag"

In [ ]:
image_link = "https://www.uoguelph.ca/research/innovation/sites/default/files/Infographics/Emily%20Heagney%20-%20Infographics%20in%20Practice.jpg"
pdf_link = "https://s21.q4cdn.com/399680738/files/doc_news/Meta-Reports-Fourth-Quarter-and-Full-Year-2025-Results-2026.pdf"
pdf_link2 = "https://s21.q4cdn.com/399680738/files/doc_financials/2026/q1/Meta-03-31-2026-Exhibit-99-1_final.pdf"
video_link = "https://samplelib.com/mp4/sample-5s-360p.mp4"
thumbnail_path = "thumbnail.png"
NINTENDO_REPORT_URL = "https://www.nintendo.co.jp/ir/pdf/2026/260203_3e.pdf"

# PDF

In [ ]:
# Extract extension
def get_extension(path: str) -> str:
    """Return the file extension from a local path or URL."""
    return pathlib.Path(path.split("?")[0]).suffix.lower()
get_extension(pdf_link)

'.pdf'

In [ ]:
# Detect Modality
def detect_modality(path: str) -> str:
    """Simple modality detection."""
    ext = get_extension(path)
    if ext in [".jpg", ".jpeg", ".png", ".webp"]:
        return "image"
    if ext == ".pdf":
        return "document"
    return "unknown"

detect_modality(pdf_link)

'document'

In [ ]:
def load_bytes(path: str) -> bytes:
    """Load bytes from a URL or local file."""
    if path.startswith("http"):
        response = requests.get(path, timeout=60)
        response.raise_for_status()
        return response.content
    return pathlib.Path(path).read_bytes()

load_bytes(pdf_link)

In [ ]:
def extract_pdf_text(path: str) -> str:
    """Extract text from a local or remote PDF."""
    pdf_bytes = load_bytes(path)
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    pages = []
    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append(f"Page {page_number}\n{text}")

    return "\n\n".join(pages)

text = extract_pdf_text(pdf_link)

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    """Split long text into overlapping chunks."""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap

    return chunks

chunks = chunk_text(text)

In [ ]:
def prepare_document(text: str, title: str = "none") -> str:
    """Format document text for retrieval."""
    return f"title: {title} | text: {text}"

In [ ]:
def embed_text(text: str) -> list[float]:
    """Embed text with Gemini Embedding 2."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )
    return result.embeddings[0].values

In [ ]:
def embed_document_text(text: str, title: str = "none") -> list[float]:
    """Embed a text chunk that we want to retrieve later."""
    return embed_text(prepare_document(text, title))

In [ ]:
ids = []
embeddings = []
documents = []
metadatas = []

for i, chunk in enumerate(chunks):
    chunk_id = f"{doc_id}_chunk_{i:03d}"
    ids.append(chunk_id)
    embeddings.append(embed_document_text(chunk, title=title))
    documents.append(chunk)
    metadatas.append({
        "parent_id": doc_id,
        "chunk": i,
        "modality": modality,
        "source": path,
        **metadata,
    })

# Images

# Videos